<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/14_a2a_protocol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 14 — A2A Protocol (the Side Step)


> **Where you are** — self-study side step. The analogy that carries the whole module: **A2A is to agents what MCP is to tools** — a discovery-plus-call protocol, one level up.
> - **First met in M02/M10, used again here:** a second process on your machine (the remote agent) that your notebook starts and talks to over HTTP.

The course finale. Thirty minutes on something that isn't strictly ADK at all — the **A2A protocol**, the industry standard for agent-to-agent communication.

You've seen agents talk to tools (M02 — MCP is the protocol for that). This module is about agents talking to **other agents** — across processes, across organizations, possibly written in completely different frameworks.

Google launched A2A at Cloud Next 2025, donated it to the Linux Foundation in June 2025, IBM's competing ACP merged in under A2A in August 2025, and v1.0 of the spec landed in early 2026. It's now the de facto standard: ~150 founding-member organizations, five language SDKs, governance under the Linux Foundation's Technical Steering Committee.

This is the single most durable thing in this course. MCP and A2A together will outlast any specific model, any specific framework, any specific vendor.


**What you'll build:**
- An ADK agent exposed over A2A via `to_a2a()` — a Starlette+uvicorn service.
- An Agent Card inspected at `/.well-known/agent-card.json`.
- A `RemoteA2aAgent` in another process that consumes the first agent as if it were local.

**What you'll leave with:**
- The four A2A nouns (Agent Card, Task, Message, Artifact).
- The A2A-vs-MCP mental model.
- The `use_legacy=False` fix that matters in production.
- A honest read on A2A's maturity (v1.0 spec; ADK integration still `@a2a_experimental`).

**Running cost:** under $0.01 (one OpenRouter call via the remote agent).

# Setup

In [1]:
!pip install -q google-adk==2.7.1 litellm==1.85.7 'a2a-sdk>=0.3.24,<0.4' uvicorn python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null
print("✅ Packages installed.")

✅ Packages installed.


In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
try: sys.stderr.fileno()
except Exception: sys.stderr = open(os.devnull, "w")

OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
    except ImportError: pass
if not OPENROUTER_API_KEY:
    from getpass import getpass
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
print("✅ Environment ready.")

✅ Environment ready.


In [3]:
import subprocess, time, tempfile, shutil, signal, json, urllib.request, asyncio, uuid
import nest_asyncio; nest_asyncio.apply()

# ADK A2A pieces
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent, AGENT_CARD_WELL_KNOWN_PATH
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

print("✅ Imports successful.")

✅ Imports successful.


# The Four A2A Nouns

A2A reduces agent-to-agent communication to four concepts. Memorize these and the rest of the protocol is commentary.

| Noun | What it is | Analogue |
|---|---|---|
| **Agent Card** | JSON descriptor served at `/.well-known/agent-card.json` — identity, capabilities, skills, auth | OpenAPI spec for an agent |
| **Task** | Stateful, server-owned unit of work. Has an ID, status, history, artifacts | A GitHub Issue, roughly |
| **Message** | One turn in a task — user or agent, with typed parts (text/file/data) | A chat message |
| **Artifact** | Durable output of a task (reports, images, structured JSON) | An Issue's attachments |

**Skills** are the semantic menu a discovering agent sees — `translate_spanish`, `find_flights`, each with id/name/description/examples. **Capabilities** are protocol feature-flags — whether the agent supports streaming, push notifications, state history.

One teaching line worth remembering: *"skills are the restaurant's menu; capabilities are whether it does delivery."*

# A2A vs MCP

The dominant framing — and it's the right one:

> **MCP is agent↔tool. A2A is agent↔agent.**

A2A's role is the agent-to-agent thing MCP isn't for. Specifically:

- **Stateful long-running tasks** — task lifecycle with `working / input-required / completed`, not just synchronous request/response.
- **Streaming + push notifications** — agents can resume after disconnection.
- **Typed artifacts** — structured outputs distinct from chat messages.
- **Peer symmetry** — every A2A agent is both client and server; no hierarchy.
- **Cross-framework interop** — an ADK agent calling a LangGraph agent calling a CrewAI agent all speaking A2A.

Where the boundary blurs: an A2A agent with a single synchronous skill is functionally indistinguishable from an MCP tool call. The distinction is **interaction model**, not what's being called.

**The layered pattern to leave this module with:** an orchestrator uses **A2A** to reach specialist agents; each specialist internally uses **MCP** to call tools. Same pattern you've built in M02+M06, with A2A as the network transport.

# Demo Part 1 — Expose an ADK Agent as A2A

The key line is one call:

```python
a2a_app = to_a2a(root_agent, port=8123)
```

The same wrapping move you've used all course — `FunctionTool` wrapped a function, `AgentTool` wrapped an agent for a *local* parent — and `to_a2a()` wraps an agent one step further: into a small web service that speaks A2A, so *any* client on the network can discover and call it. Under the hood it builds a Starlette app; you run it with `uvicorn` like any Python web app.

We'll write a small specialist agent to a temp directory, then launch it as a subprocess on `localhost:8123`. The agent does one thing: convert Celsius to Fahrenheit via a tool call.

In [4]:
# Write the specialist agent + server script to a temp dir
SRV_DIR = tempfile.mkdtemp(prefix="adk_m14_a2a_")
SRV_SCRIPT = os.path.join(SRV_DIR, "server.py")

SERVER_CODE = (
    "import os\n"
    "from dotenv import load_dotenv\n"
    "load_dotenv()\n\n"
    "from google.adk.agents import LlmAgent\n"
    "from google.adk.models.lite_llm import LiteLlm\n"
    "from google.adk.a2a.utils.agent_to_a2a import to_a2a\n"
    "import uvicorn\n\n"
    "def convert_c_to_f(celsius: float) -> dict:\n"
    "    'Convert Celsius to Fahrenheit.'\n"
    "    return {'celsius': celsius, 'fahrenheit': round(celsius * 9/5 + 32, 2)}\n\n"
    "root_agent = LlmAgent(\n"
    "    name='temperature_specialist',\n"
    "    model=LiteLlm(model='openrouter/openai/gpt-5.6-luna'),\n"
    "    description='Converts Celsius to Fahrenheit via convert_c_to_f tool.',\n"
    "    instruction='Convert temperatures using the tool. Return only the number.',\n"
    "    tools=[convert_c_to_f],\n"
    ")\n\n"
    "app = to_a2a(root_agent, host='localhost', port=8123)\n"
    "uvicorn.run(app, host='localhost', port=8123, log_level='error')\n"
)

with open(SRV_SCRIPT, "w") as f:
    f.write(SERVER_CODE)

# Launch server as subprocess
env = {**os.environ}
proc = subprocess.Popen(
    [sys.executable, SRV_SCRIPT],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env=env,
)
time.sleep(7)  # give it time to start uvicorn + ADK
print(f"✅ A2A server running (PID {proc.pid}) on http://localhost:8123")

✅ A2A server running (PID 6059) on http://localhost:8123


## Fetch the Agent Card

The Agent Card is the A2A discovery mechanism. Served at `/.well-known/agent-card.json` (this path was renamed from `/.well-known/agent.json` in A2A v0.3.0 — a trap for anyone reading older blogs). A2A clients fetch it to learn:

- The agent's name, description, skills.
- The transport preference (JSON-RPC, gRPC, or REST).
- The protocol version, capabilities, supported modalities.
- Security schemes (OAuth, API key, mTLS, etc.).

In [5]:
with urllib.request.urlopen(f"http://localhost:8123{AGENT_CARD_WELL_KNOWN_PATH}") as r:
    card = json.loads(r.read().decode())

print(json.dumps(card, indent=2)[:2000])

{
  "capabilities": {
    "pushNotifications": false,
    "streaming": false
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "description": "Converts Celsius to Fahrenheit via convert_c_to_f tool.",
  "name": "temperature_specialist",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "skills": [
    {
      "description": "Converts Celsius to Fahrenheit via convert_c_to_f tool.",
      "examples": [],
      "id": "temperature_specialist",
      "name": "model",
      "tags": [
        "llm"
      ]
    },
    {
      "description": "Convert Celsius to Fahrenheit.",
      "id": "temperature_specialist-convert_c_to_f",
      "name": "convert_c_to_f",
      "tags": [
        "llm",
        "tools"
      ]
    }
  ],
  "supportsAuthenticatedExtendedCard": false,
  "url": "http://localhost:8123",
  "version": "0.0.1"
}


The card is auto-generated from the agent's `name` + `description` + tools. `preferredTransport` is JSON-RPC by default (works over HTTPS; widely supported). `protocolVersion` is 0.3.0 — that's what ADK's `a2a-sdk 0.3.x` dependency speaks (still true on the ADK 2.x line, which pins `a2a-sdk >=0.3.4,<0.4`).

In production you'd author the card more carefully — populate `skills[]` with specific examples, declare `capabilities.streaming: true` if your agent supports it, add signed identity (`AgentCardSignature`). For a course demo, the auto-generated version is enough.

### 🎯 Mini-task

Author the card by hand and pass it via `to_a2a(agent_card=...)`: add `examples` to `skills`, declare `capabilities.streaming`. Does the demo still work with your hand-written card?

# Demo Part 2 — Consume the Agent as a `RemoteA2aAgent`

`RemoteA2aAgent` is the consuming side of the same idea: give it an Agent Card URL, it fetches the card, and from then on the remote agent behaves like a local `LlmAgent` object. Drop it into any runner's agent argument, or into another agent's `sub_agents=` list — from the consuming side, it looks identical to a local agent.

**Critical**: pass `use_legacy=False`. The legacy streaming path has three known bugs (user-message duplication, remote outputs mis-classified as thoughts, sub-agent output loss). The new path replaces the executor with one that fixes these. `False` is the correct choice for new code.

In [6]:
remote = RemoteA2aAgent(
    name="remote_temp_agent",
    agent_card=f"http://localhost:8123{AGENT_CARD_WELL_KNOWN_PATH}",
    description="Remote temperature conversion specialist.",
    use_legacy=False,   # critical; skips the three known streaming bugs
)

print("✅ RemoteA2aAgent ready.")
print(f"   Pointing at: http://localhost:8123{AGENT_CARD_WELL_KNOWN_PATH}")

✅ RemoteA2aAgent ready.
   Pointing at: http://localhost:8123/.well-known/agent-card.json


In [7]:
# Call the remote agent locally via Runner — same API as any other LlmAgent
session_service = InMemorySessionService()
await session_service.create_session(app_name="m14", user_id="student", session_id="s1")
runner = Runner(agent=remote, app_name="m14", session_service=session_service)

msg = types.Content(role="user", parts=[types.Part(text="Convert 20 degrees Celsius to Fahrenheit.")])
print("USER: Convert 20 degrees Celsius to Fahrenheit.\n")

async for ev in runner.run_async(user_id="student", session_id="s1", new_message=msg):
    if ev.content and ev.content.parts:
        for p in ev.content.parts:
            if p.text and p.text.strip():
                print(f"[{ev.author}] {p.text.strip()[:200]}")
            if p.function_call:
                print(f"[tool_call] {p.function_call.name}")
            if p.function_response:
                print(f"[tool_resp] {p.function_response.response}")

USER: Convert 20 degrees Celsius to Fahrenheit.



[tool_call] convert_c_to_f
[tool_resp] {'celsius': 20, 'fahrenheit': 68.0}
[remote_temp_agent] 68.0


### 🔍 What just happened?

Read the event stream carefully. **The tool call (`convert_c_to_f`) executed on the remote server's side of the WebSocket.** Your side sees it as a regular event — tool call, tool response, final text — just like if the agent were local.

This is what A2A buys you. From your agent's perspective, the specialist might as well be local. From the network's perspective, you just made an HTTP call to a separate process (or a separate machine, or a separate organization). The A2A protocol handles the translation.

**Remember the two multi-agent patterns?** `sub_agents` for transfer, `AgentTool` for consultant calls. `RemoteA2aAgent` plugs into either — it works as a child in `sub_agents=[RemoteA2aAgent(...)]`, or as the `agent=` argument of an `AgentTool`. Same composition patterns, A2A as the transport underneath.

### 🎯 Mini-tasks

1. **Cross-framework A2A.** Clone [`a2aproject/a2a-samples`](https://github.com/a2aproject/a2a-samples), run the LangGraph currency-converter sample on port 8124, and consume it with a `RemoteA2aAgent` from this notebook. Does ADK talk to LangGraph cleanly?
2. **Two-agent orchestrator.** Keep the temperature specialist running, add a second remote specialist (string reversal), and build a local orchestrator with BOTH in `sub_agents=`. Does it route correctly?
3. **See the wire format.** Capture port 8123 with `mitmproxy` and look at the JSON-RPC messages. What does a `message/stream` call look like?

# Cleanup

In [8]:
proc.terminate()
try:
    proc.wait(timeout=3)
except subprocess.TimeoutExpired:
    proc.kill()
shutil.rmtree(SRV_DIR, ignore_errors=True)
print("✅ Server stopped, temp dir cleaned.")

✅ Server stopped, temp dir cleaned.


# A2A Maturity — An Honest Read

Worth naming. A2A's trajectory:

- **April 2025** — Google launches at Cloud Next. Spec v0.1.
- **June 2025** — Donated to the Linux Foundation. Technical Steering Committee forms.
- **August 2025** — IBM's competing ACP merged in.
- **December 2025** — MCP donated to the new Agentic AI Foundation (under the Linux Foundation); A2A and MCP under cross-vendor governance.
- **Early 2026** — A2A v1.0 spec landed. Five official language SDKs (Python, JavaScript, Java, Go, .NET).
- **Now** — ~150 founding-member organizations. ~23K GitHub stars. ADK integration still marked `@a2a_experimental`.


What this means in practice:

- **The spec is real.** You can read A2A v1.0 and build against it knowing the shape will be stable.
- **The ecosystem is thin.** Most of those 150 orgs are signatories, not shipping production integrations. Named customer references exist (Adobe, Tyson Foods, S&P Global) but no deep post-mortems.
- **The ADK integration is experimental.** You'll see `@a2a_experimental` warnings. `use_legacy=False` exists because the legacy path has bugs. Version-pin carefully: ADK (through 2.7) pins `a2a-sdk` to 0.3.x; the a2a-sdk 1.x line breaks against it.

The honest framing: **A2A is architecture worth understanding, not infrastructure you'd bet production on yet.** Build against it for new work; don't migrate existing production workloads yet. By late 2026 this should invert as the ecosystem matures.

# Gotchas Worth Knowing

Six sharp edges to pre-empt:

1. **The `.well-known` path rename.** `/.well-known/agent.json` in v0.2 became `/.well-known/agent-card.json` in v0.3. Code copied from older blog posts WILL be wrong.
2. **Legacy executor bugs.** `RemoteA2aAgent(..., use_legacy=True)` (the default) has three issues: user-message duplication, remote outputs mis-classified as thoughts, sub-agent output loss. **Pass `use_legacy=False`.**
3. **Version-pin ADK and a2a-sdk together.** ADK (through 2.7) requires `a2a-sdk >=0.3.4,<0.4`; A2A v1.0 requires `a2a-sdk ≥ 1.0` which ADK doesn't yet speak. Don't mix.
4. **Discovery is underspecified.** The spec defines `.well-known` as one mechanism; registries are explicitly "future exploration." Don't build on registry features.
5. **Agent Engine is non-spec-default.** Google's Agent Engine serves the Agent Card behind authentication at `/v1/card`, not at `/.well-known/`. Third-party A2A clients expecting the standard path will fail against Agent Engine.
6. **Cross-org trust is unsolved.** Signed Agent Cards are in v1.0 as SHOULD, not MUST. There's no central root-of-trust CA. Treat every cross-org A2A response as untrusted input to your planner — **the lethal trifecta from Simon Willison's framing applies fully to A2A**.

# Key Takeaways — M14

- **Four A2A nouns**: Agent Card, Task, Message, Artifact. Memorize these.
- **A2A is agent↔agent; MCP is agent↔tool.** The layered pattern: orchestrator uses A2A; specialists use MCP.
- **`to_a2a(agent)`** exposes any ADK agent as a Starlette A2A server. **`RemoteA2aAgent(agent_card=...)`** consumes one from any process.
- **`use_legacy=False`** is non-negotiable for new `RemoteA2aAgent` code — the legacy executor has three known bugs.
- **Agent Cards** are served at `/.well-known/agent-card.json` — auto-generated by ADK, but hand-authorable for production.
- **Version-pin** `google-adk` and `a2a-sdk` together. ADK (through 2.7) speaks the 0.3.x protocol.
- **Maturity check**: A2A is architecture worth understanding, not production infrastructure yet.
- **Cross-org trust is unsolved**; every A2A response from outside your org is untrusted input.


# Course finale

Fourteen modules. From "what is an agent?" to "here's how agents talk to each other across organizations."

## Part 1 — Vendor-agnostic spine (M01-M10)
The four primitives; four tool flavors; state with scope prefixes; one-line model swaps; workflow agents (Sequential / Parallel / Loop); multi-agent via sub_agents and AgentTool; callbacks as your code around every step; memory with persistence and long-term recall; automated eval; deployment.

## Part 2 — Gemini unlocks (M11-M13)
Google Search grounding with real citations; long context + 90%-discount caching; thinking budgets; Live API voice.

## Side step (M14)
A2A protocol for cross-framework agent-to-agent communication.

**What to do next:**
- Build something. A voice-first customer-support bot. A research orchestrator. A personal assistant that survives restarts. The course taught the mechanics; building teaches the rest.
- Follow the spec. A2A v1.0 and MCP are both under Linux Foundation governance now. The protocols will outlast ADK-as-a-framework; invest in understanding them as durable.
- Watch the course repo. `DEMOS_BROKEN.md` will be updated as preview APIs stabilize and fragile demos start working.

Thanks for taking the course. Go build something.